In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import requests

# Define the URLs and the local file paths
urls = [
    "https://public.fyers.in/sym_details/NSE_FO.csv",
    "https://public.fyers.in/sym_details/NSE_CM.csv",
    "https://public.fyers.in/sym_details/BSE_FO.csv"
]
local_file_paths = [
    "/content/NSE_FO.csv",
    "/content/NSE_CM.csv",
    "/content/BSE_FO.csv"
]

# Download the CSV files from the URLs
for url, local_file_path in zip(urls, local_file_paths):
    response = requests.get(url)
    response.raise_for_status()  # Ensure we notice bad responses

    # Save the CSV file to the specified local path
    with open(local_file_path, 'wb') as file:
        file.write(response.content)

    print(f"File downloaded and saved to {local_file_path}")


File downloaded and saved to /content/NSE_FO.csv
File downloaded and saved to /content/NSE_CM.csv
File downloaded and saved to /content/BSE_FO.csv


In [ ]:
input_date = '24 Jun'

In [ ]:
csv_path = 'NSE_FO.csv'
p=csv_path

In [ ]:
print(p)

NSE_FO.csv


In [ ]:
import pandas as pd
import re

# Path to your input CSV file


# Path to your output CSV file
output_csv_path = '/content/u1.csv'

# Reading the CSV file with specified column names
column_names = [
    'id', 'description', 'unknown2', 'lot_size', 'unknown4', 'unknown5',
    'unknown6', 'market_times', 'date', 'formatted_symbol', 'unknown10',
    'unknown11', 'ref_id', 'symbol', 'contract_size', 'strike',
    'option_type', 'contract_code', 'unknown17', 'unknown18', 'unknown19'
]
broker_data = pd.read_csv(csv_path, header=None, names=column_names)

# Extracting values from the 'description' column that match the pattern
extracted_descriptions = []
pattern = re.compile(rf'^(.*{input_date} \d{{2}})')

for desc in broker_data['description']:
    match = pattern.match(desc)
    if match:
        extracted_descriptions.append(match.group(1))

# Finding unique values
unique_descriptions = list(set(extracted_descriptions))

# Sorting the unique descriptions with 'BANKEX' first and 'SENSEX' after
sorted_descriptions = sorted(unique_descriptions, key=lambda x: ('SENSEX' in x, x))

# Saving the sorted unique descriptions to a new CSV file
sorted_descriptions_df = pd.DataFrame(sorted_descriptions, columns=['unique_description'])
sorted_descriptions_df.to_csv(output_csv_path, index=False)

print(f"Unique descriptions have been saved to {output_csv_path}")


Unique descriptions have been saved to /content/u1.csv


In [ ]:
broker_data.head()

,id,description,unknown2,lot_size,unknown4,unknown5,unknown6,market_times,date,formatted_symbol,...,unknown11,ref_id,symbol,contract_size,strike,option_type,contract_code,unknown17,unknown18,unknown19
0,101124052835089,FINNIFTY 24 May 28 FUT,11,40,0.05,NaN,0915-1530|1815-1915:,2024-05-27,1716890400,NSE:FINNIFTY24MAYFUT,...,11,35089,FINNIFTY,26037,-1.0,XX,101000000026037,NaN,0,0.0
1,101124052835129,FINNIFTY 24 May 28 26900 CE,14,40,0.05,NaN,0915-1530|1815-1915:,2024-05-27,1716890400,NSE:FINNIFTY24MAY26900CE,...,11,35129,FINNIFTY,26037,26900.0,CE,101000000026037,NaN,0,0.0
2,101124052835137,FINNIFTY 24 May 28 26900 PE,14,40,0.05,NaN,0915-1530|1815-1915:,2024-05-27,1716890400,NSE:FINNIFTY24MAY26900PE,...,11,35137,FINNIFTY,26037,26900.0,PE,101000000026037,NaN,0,0.0
3,101124052835477,FINNIFTY 24 May 28 16100 CE,14,40,0.05,NaN,0915-1530|1815-1915:,2024-05-27,1716890400,NSE:FINNIFTY24MAY16100CE,...,11,35477,FINNIFTY,26037,16100.0,CE,101000000026037,NaN,0,0.0
4,101124052835478,FINNIFTY 24 May 28 16100 PE,14,40,0.05,NaN,0915-1530|1815-1915:,2024-05-27,1716890400,NSE:FINNIFTY24MAY16100PE,...,11,35478,FINNIFTY,26037,16100.0,PE,101000000026037,NaN,0,0.0


In [ ]:
import pandas as pd

# Paths to your input CSV files

u1_csv_path = '/content/u1.csv'
output_csv_path = 'm3.csv'

# Reading the new CSV file containing unique descriptions
u1_data = pd.read_csv(u1_csv_path, header=None, names=['unique_description'])

# Reading the original CSV file with specified column names
column_names = [
    'id', 'description', 'unknown2', 'lot_size', 'unknown4', 'unknown5',
    'unknown6', 'market_times', 'date', 'formatted_symbol', 'unknown10',
    'unknown11', 'ref_id', 'symbol', 'contract_size', 'strike',
    'option_type', 'contract_code', 'unknown17', 'unknown18', 'unknown19'
]
broker_data = pd.read_csv(csv_path, header=None, names=column_names)

# Initialize a list to store the results
results = []

# Iterate through each unique description in u1.csv
for unique_description in u1_data['unique_description']:
    # Find rows in broker_data where the description starts with the unique description
    matched_rows = broker_data[broker_data['description'].str.startswith(unique_description)]

    # Filter out rows where formatted_symbol ends with 'FUT' and ensure there is a valid strike value
    matched_rows = matched_rows[~matched_rows['formatted_symbol'].str.endswith('FUT') & matched_rows['strike'].notna()]

    # If there are any valid matched rows, take the first one
    if not matched_rows.empty:
        row = matched_rows.iloc[0]
        results.append({
            'unique_description': unique_description,
            'formatted_symbol': row['formatted_symbol'],
            'strike': row['strike']
        })

# Convert the results to a DataFrame
results_df = pd.DataFrame(results)

# Save the results to a new CSV file
results_df.to_csv(output_csv_path, index=False)

print(f"Extracted data has been saved to {output_csv_path}")


Extracted data has been saved to m3.csv


In [ ]:
import pandas as pd
import re

# Read the CSV file
file_path = "m3.csv"
df = pd.read_csv(file_path)

# Function to process each row based on the rules provided
def process_row(row):
    formatted_symbol = row['formatted_symbol']

    # Find the first occurrence of a digit in the formatted_symbol
    match = re.search(r'\d', formatted_symbol)
    if match:
        # Extract and return the part from the first digit to the end
        result = formatted_symbol[match.start():]
        return result
    else:
        # If no digit is found, return the original formatted_symbol
        return formatted_symbol

# Apply the function to each row
df['processed'] = df.apply(process_row, axis=1)

# Save the processed data to a new CSV file
output_file_path = "y1.csv"
df.to_csv(output_file_path, index=False)

print(f"Processed data saved to {output_file_path}")


Processed data saved to y1.csv


In [ ]:
#

In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('y1.csv')

# Function to remove suffixes
def remove_suffix(text):
    return text.replace('CE', '').replace('PE', '')

# Apply the function to the 'processed' column and create a new 'cleaned' column
df['cleaned'] = df['processed'].apply(remove_suffix)

# Save the updated DataFrame to a new CSV file
df.to_csv('y10.csv', index=False)

print("New CSV file 'y1_cleaned.csv' has been created with the cleaned data.")


New CSV file 'y1_cleaned.csv' has been created with the cleaned data.


In [ ]:
#

In [ ]:
import pandas as pd

# Read the CSV file
csv_path = 'y10.csv'
df = pd.read_csv(csv_path)

# Function to remove strike value from cleaned column
def remove_strike(row):
    strike = str(int(row['strike']))  # Convert strike to string without decimal
    cleaned = str(row['cleaned'])
    return cleaned.replace(strike, '')

# Apply the function to each row and create a new 'syntax' column
df['syntax'] = df.apply(remove_strike, axis=1)

# Save the updated DataFrame to a new CSV file
output_csv_path = 'j.csv'
df.to_csv(output_csv_path, index=False)

print(f"New CSV file '{output_csv_path}' has been created with the syntax column.")


New CSV file 'j.csv' has been created with the syntax column.


In [ ]:
import pandas as pd

# Define the symbols to match
symbols = ['NIFTY', 'BANKNIFTY', 'FINNIFTY', 'MIDCPNIFTY', 'NIFTYNXT50']

# Read the CSV file
csv_path = 'j.csv'
df = pd.read_csv(csv_path)

# Filter rows where 'unique_description' starts with any of the specified symbols
filtered_df = df[df['unique_description'].str.startswith(tuple(symbols))]

# Save the filtered data to a new CSV file
output_csv_path = 'j2.csv'
filtered_df.to_csv(output_csv_path, index=False)

print(f"Filtered data has been saved to {output_csv_path}")


Filtered data has been saved to j2.csv


In [ ]:
#D

In [ ]:
import pandas as pd
from datetime import datetime

# Read the CSV file
csv_path = 'j2.csv'
df = pd.read_csv(csv_path)

# Function to extract and parse the date
def extract_date(description):
    parts = description.split()
    day = parts[-1]     # The day is the last part
    month = parts[-2]   # The month is the second last part
    year = '20' + parts[-3]  # The year is the third last part, prepended with '20'
    date_str = f"{day} {month} {year}"
    return datetime.strptime(date_str, '%d %b %Y')

# Apply the function to extract and parse dates
df['date'] = df['unique_description'].apply(extract_date)

# Sort the DataFrame by the parsed dates
sorted_df = df.sort_values(by='date')

# Drop the temporary 'date' column
sorted_df = sorted_df.drop(columns=['date'])

# Save the sorted data to a new CSV file
output_csv_path = 'JACK5.csv'
sorted_df.to_csv(output_csv_path, index=False)

print(f"Sorted data has been saved to {output_csv_path}")


Sorted data has been saved to JACK5.csv


In [ ]:
import pandas as pd

# Read the CSV file
csv_path = 'JACK5.csv'  # Update with your file path
df = pd.read_csv(csv_path)

# Select only the 'unique_description' and 'syntax' columns
selected_columns_df = df[['unique_description', 'syntax']]

# Reset the index to start from 1
selected_columns_df.index = selected_columns_df.index + 1

# Save the selected columns to a new CSV file
output_csv_path = '1CALENDER.csv'  # Update with your desired output path
selected_columns_df.to_csv(output_csv_path, index_label='index')

print(f"Selected columns have been saved to {output_csv_path}")


Selected columns have been saved to 1CALENDER.csv


In [ ]:
import pandas as pd

# Read the CSV file into a DataFrame
df = pd.read_csv(p, header=None)



In [ ]:
symbols = ['NIFTY', 'BANKNIFTY', 'FINNIFTY', 'MIDCPNIFTY', 'NIFTYNXT50']

# Filter rows containing the specified symbols and the suffix "24 May" in the second column
filtered_rows = df[df[1].str.split(' ').str[0].isin(symbols) & df[1].str.contains(input_date)]

# Extracting the required data from the second column
filtered_rows['Extracted'] = filtered_rows[1].str.split(' ').str[:4].str.join(' ')

# Remove duplicates
extracted_unique = filtered_rows['Extracted'].unique()

# Create a dictionary to store dates for each symbol
data = {symbol: [] for symbol in symbols}

# Fill the dictionary with dates corresponding to each symbol
for symbol in symbols:
    for extracted_value in extracted_unique:
        if extracted_value.startswith(symbol):
            data[symbol].append(extracted_value)

# Find the maximum number of dates for any symbol
max_length = max(len(data[symbol]) for symbol in symbols)

# Pad lists with NaNs for symbols with fewer dates
for symbol in symbols:
    data[symbol] += [float('nan')] * (max_length - len(data[symbol]))

# Convert the dictionary to a DataFrame
result_df = pd.DataFrame(data)

# Write the DataFrame to a CSV file
result_df.to_csv('1BOX.csv', index=False)

<ipython-input-39-3a840e057ac5>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_rows['Extracted'] = filtered_rows[1].str.split(' ').str[:4].str.join(' ')


In [ ]:
#

In [40]:
import pandas as pd

# Read the data from a.csv
df_a = pd.read_csv('/content/MAYCALENDER.csv')

# Read the data from b.csv
df_b = pd.read_csv('/content/1CALENDER.csv')

# Combine the data from both DataFrames
combined_df = pd.concat([df_a, df_b], ignore_index=True)

# Save the combined data to a new CSV file
combined_df.to_csv('combined.csv', index=False)

print("Data from a.csv and b.csv has been successfully combined and saved to combined.csv")


Data from a.csv and b.csv has been successfully combined and saved to combined.csv
